In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import trimesh
import fvdb
import argparse
from pprint import pprint
import tqdm


# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
import eval
import utils
import models
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 

In [2]:
# read config file
config = read_config.read_yaml_config(f'config_04072025.yaml')
print("Configuration loaded:")
for key, value in config.items():
    pprint(f"{key}: {value}")

# # initialize logging
# logger = wandb_logging.WandbLogger(
#                     logging=config['logging'],
#                     project_name=config['wandb']['project_name'],
#                     entity=config['wandb']['entity'],
#                     name=config['wandb']['name'],
#                     group=config['wandb']['group'],
#                     tags=config['wandb']['tags'],
#                     notes=config['wandb']['notes'],
#                     config=config['wandb']['config']
#                 )


Configuration loaded:
'logging: True'
"reproducibility: {'is_reproducible': True, 'seed': 42}"
("data: {'input_dir': "
 "'/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt', "
 "'data_description': 'Generated SDF of 32, 64, 128 data using NMC logic', "
 "'data_name': 'V1_Manual_Extracted_NMC_Data', 'data_split': {'train': 0.6, "
 "'val': 0.2, 'test': 0.2}, 'mask_threshold': 33, 'is_crop': {'train': True, "
 "'val': False, 'test': False}, 'crop_size': 16, 'crop_threshold': 100, "
 "'n_jobs': -1, 'batch_size': 32, 'shuffle': {'train': True, 'val': False, "
 "'test': False}, 'num_workers': 0}")
("training: {'save_model_dir': "
 "'/data/workspaces/spanwar/results/ssu/save_models', 'model_name': "
 "'27_Unet_Upsampler_Manual_NMC_data', 'lr': 0.001, 'epochs': 10, "
 "'loss_function': 'mse_loss', 'optimizer': 'Adam', 'loss_weights': [0.25, "
 "0.25, 0.5], 'save_model': True, 'positional_encoding': 10, 'in_channels': 1, "
 "'out_channels': 1}")
(

In [3]:
# config['eval']['only_eval'] = True
# set reproducibility
st.set_reproducibility(is_reproducible=config['reproducibility']['is_reproducible'],
                        seed=config['reproducibility']['seed'])

# load data
input_dir = config['data']['input_dir']

# load data
input_dir = config['data']['input_dir']
(train_dataloader, 
    val_dataloader, 
    test_dataloader) = ABC_dataset_loader.ABCDataLoader(
                                    input_dir=input_dir,
                                    config=config,
                                    # n_samples=10
                                ).get()


Setting reproducibility with seed: 42
Dataset split: 3210 train, 1070 val, 1071 test


100%|██████████| 1071/1071 [00:15<00:00, 67.00it/s] 


In [5]:
# # get all the test files
# all_test_files = []
# for batch in tqdm.tqdm(test_dataloader):
#     file_name, _,_,_ = batch
#     file_name = file_name[0]
#     all_test_files.append(file_name)

# all_test_files

In [6]:
# # Save the results to a txt file
# with open('test_names_file.txt', 'w') as f:
#     for file_name in all_test_files:
#         f.write(f"{file_name}\n")

In [9]:

# Get the first batch from the DataLoader
first_batch = next(iter(train_dataloader))

# If your DataLoader returns a tuple (vdb_32, vdb_64, vdb_128), each is a batch of 32
names, vdb_32_batch, vdb_64_batch, vdb_128_batch = first_batch

# Get the first sample from each batch
name = names[0]
vdb_32 = vdb_32_batch[0]
vdb_64 = vdb_64_batch[0]
vdb_128 = vdb_128_batch[0]
sdfToVDB = fu.sdfToVDB(threshold=config['data']['mask_threshold'])


In [61]:
sdfToVDB.plot_vdb(vdb_32)
sdfToVDB.plot_vdb(vdb_64)
sdfToVDB.plot_vdb(vdb_128)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0614189…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0012528…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0012686…

In [10]:
name = '00007347'
# Get the first batch from the DataLoader
for first_batch in test_dataloader:
    # If your DataLoader returns a tuple (vdb_32, vdb_64, vdb_128), each is a batch of 32
    names, vdb_32_batch, vdb_64_batch, vdb_128_batch = first_batch

    # Check if the name exists in the batch
    if f'{name}.hdf5' in names:
        break

# If your DataLoader returns a tuple (vdb_32, vdb_64, vdb_128), each is a batch of 32
names, vdb_32_batch, vdb_64_batch, vdb_128_batch = first_batch


In [13]:
names

['00007347.hdf5']

In [14]:

# Get the first sample from each batch
# find index of specific name in names
# index = names.index(f'{name}.hdf5')
index=0
name_ = names[index]
print(name_, index, name)
vdb_32 = vdb_32_batch[index]
vdb_64 = vdb_64_batch[index]
vdb_128 = vdb_128_batch[index]
sdfToVDB = fu.sdfToVDB(threshold=config['data']['mask_threshold'])
sdfToVDB.plot_vdb(vdb_32)
sdfToVDB.plot_vdb(vdb_64)
sdfToVDB.plot_vdb(vdb_128)


00007347.hdf5 0 00007347


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0614189…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0012528…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0012686…

In [70]:
# v, f from fvdb sdf
v, f = sdfToVDB.vdb_marching_cubes(vdb_128)
mesh_training_128 = trimesh.Trimesh(vertices=v, faces=f)
mesh_training_128.export('data/training_128_' + name.split('.')[0] + '.PLY')  # Save as PLY file

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 2798\nproperty float x\nproperty float y\nproperty float z\nelement face 5592\nproperty list uchar int vertex_indices\nend_header\n\xa0m\x0c\xbc\x00\x00\xa0\xbd\x00\x00\xf4\xbe\x00\x00\x00\xbc9x\xa3\xbd\x00\x00\xf4\xbe\x00\x00\x00\xbc\x00\x00\xa0\xbd\xa2h\xf7\xbe\xef1\x14\xbc\x00\x00\xa0\xbd\x00\x00\xf0\xbe\x00\x00\x00\xbc\x15\xc3\xa5\xbd\x00\x00\xf0\xbe\xde\xc2 \xbc\x00\x00\xa0\xbd\x00\x00\xec\xbe\x00\x00\x00\xbc\xb6\x84\xa8\xbd\x00\x00\xec\xbe\xf9u(\xbc\x00\x00\xa0\xbd\x00\x00\xe8\xbe\x00\x00\x00\xbcy\xac\xaa\xbd\x00\x00\xe8\xbe\xbcV0\xbc\x00\x00\xa0\xbd\x00\x00\xe4\xbe\x00\x00\x00\xbc\x02A\xad\xbd\x00\x00\xe4\xbe\x00\x00\x00\xbc\xb4\xc5\x9b\xbd\x00\x00\xf4\xbe\x00\x00\x00\xbc\xcc!\x99\xbd\x00\x00\xf0\xbe\x00\x00\x00\xbc>v\x95\xbd\x00\x00\xec\xbe\x00\x00\x00\xbc\xd6\x1a\x93\xbd\x00\x00\xe8\xbe\x00\x00\x00\xbc\x915\x91\xbd\x00\x00\xe4\xbe\xaa\x9b\x0c\xbc\x00\x00\xb0\xbd\x00\x00\xdc\xbe\x0

In [64]:
# nvdb to mesh
prediction_dir = config['eval']['save_predictions_dir']
os.listdir(prediction_dir)

['SSU_PONQ_DATA_UPSAMPLER_Unet',
 '26_rerun_24_SSU_PONQ_DATA_UPSAMPLER_FM_trilinear_Unet_v2',
 '27_Unet_Upsampler_Manual_NMC_data',
 '28_Unet_Upsampler_Manual_NMC_data_l1',
 '29_Unet_Upsampler_Manual_NMC_data_l1',
 '30_Unet_Upsampler_Manual_NMC_data_l1',
 '31_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '32_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '33_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '34_Simple_CNN_Upsampler_Manual_NMC_data_huber_loss',
 '35_rerun_33_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '36_rerun_35_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '37_residual_CNN_Upsampler_Manual_NMC_data_l1']

In [14]:
name

'00008322.hdf5'

In [71]:
# pred_dir_27 = os.path.join(prediction_dir, '27_Unet_Upsampler_Manual_NMC_data')
# pred_dir_28 = os.path.join(prediction_dir, '28_Unet_Upsampler_Manual_NMC_data_l1')
def get_test_prediction(obj_name, model_name):
    pred_dir = os.path.join(prediction_dir, model_name)
    pred_vdb = fvdb.load(os.path.join(pred_dir, 
                                     str(config['eval']['upsampling_level']), 
                                     obj_name.split('.')[0] + '.nvdb'))
    v, f, _ = pred_vdb[0].marching_cubes(pred_vdb[1])
    mesh = trimesh.Trimesh(vertices=v.jdata.numpy(), faces=f.jdata.numpy())
    return mesh
mesh = get_test_prediction(name, '31_Simple_CNN_Upsampler_Manual_NMC_data_l1')
mesh.export('data/31_' + name.split('.')[0] + '.PLY')  # Save as PLY file

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 2850\nproperty float x\nproperty float y\nproperty float z\nelement face 5696\nproperty list uchar int vertex_indices\nend_header\n\xf4\x18\x03\xbc\x00\x00\xb0\xbd\x00\x00\xe4\xbe\x00\x00\x00\xbc\xd9V\xb0\xbd\x00\x00\xe4\xbe\x00\x00\x00\xbc\x00\x00\xb0\xbd\xbc\xc4\xe4\xbe\xd0{\x0b\xbc\x00\x00\xa0\xbd\x00\x00\xf4\xbe\x00\x00\x00\xbc\xd9\x0e\xa2\xbd\x00\x00\xf4\xbe\x00\x00\x00\xbc\x00\x00\xa0\xbdY\xcc\xf5\xbe\x02[\x19\xbc\x00\x00\xa0\xbd\x00\x00\xf0\xbe\x00\x00\x00\xbc8b\xa5\xbd\x00\x00\xf0\xbe\xde\x147\xbc\x00\x00\xa0\xbd\x00\x00\xec\xbe\x00\x00\x00\xbc\xa6\x0f\xac\xbd\x00\x00\xec\xbe\xfeVA\xbc\x00\x00\xa0\xbd\x00\x00\xe8\xbe\x00\x00\x00\xbc\xb3\xfa\xad\xbd\x00\x00\xe8\xbe\xce\xffK\xbc\x00\x00\xa0\xbd\x00\x00\xe4\xbe\x00\x00\x00\xbct:\x9d\xbd\x00\x00\xf4\xbe\x00\x00\x00\xbc\x0cz\x98\xbd\x00\x00\xf0\xbe\x00\x00\x00\xbc\xb9\\\x91\xbd\x00\x00\xec\xbe\xe4v\x0f\xbc\x00\x00\x90\xbd\x00\x00\xe8\xb

In [20]:
pred_vdb_27 = fvdb.load(os.path.join(pred_dir_27, 
                                     str(config['eval']['upsampling_level']), 
                                     name.split('.')[0] + '.nvdb'))
pred_vdb_28 = fvdb.load(os.path.join(pred_dir_28, 
                                     str(config['eval']['upsampling_level']),
                                     name.split('.')[0] + '.nvdb'))

In [21]:
v, f, _ = pred_vdb_27[0].marching_cubes(pred_vdb_27[1])
mesh_27 = trimesh.Trimesh(vertices=v.jdata.numpy(), faces=f.jdata.numpy())
v, f, _ = pred_vdb_28[0].marching_cubes(pred_vdb_28[1])
mesh_28 = trimesh.Trimesh(vertices=v.jdata.numpy(), faces=f.jdata.numpy())

In [26]:
# save meshes to obj files
mesh_27.export('data/27_' + name.split('.')[0] + '.PLY')  # Save as OBJ file
mesh_28.export('data/28_' + name.split('.')[0] + '.PLY')  # Save as PLY file

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 13892\nproperty float x\nproperty float y\nproperty float z\nelement face 27780\nproperty list uchar int vertex_indices\nend_header\nO\xb7\x08\xbe\x00\x00\xec\xbe\x00\x00\xc0\xbd\x00\x00\x08\xbe\x0e\x0b\xed\xbe\xff\xff\xbf\xbd\x00\x00\x08\xbe\x00\x00\xec\xbe\x81\x19\xc6\xbdv\xcb\x08\xbe\x00\x00\xec\xbe\x00\x00\xb0\xbd\x00\x00\x08\xbe\x0e\x88\xec\xbe\x00\x00\xb0\xbdg<\x08\xbe\x00\x00\xec\xbe\x00\x00\xa0\xbd\x00\x00\x08\xbe\xe6@\xec\xbe\x00\x00\xa0\xbd\x00\x00\x08\xbe\x00\x00\xec\xbe\x00\x88\x91\xbd\x96\xa7\t\xbe\x00\x00\xe8\xbe\x00\x00\xc0\xbd\x00\x00\x08\xbe\x00\x00\xe8\xbe\xa93\xc7\xbdB\xd7\t\xbe\x00\x00\xe8\xbe\x00\x00\xb0\xbd\x1b\xfd\t\xbe\x00\x00\xe8\xbe\x00\x00\xa0\xbd2S\t\xbe\x00\x00\xe8\xbe\x00\x00\x90\xbd\x00\x00\x08\xbe?\xf1\xeb\xbe\x00\x00\x90\xbd\x97\xbc\t\xbe\x00\x00\xe4\xbe\x00\x00\xc0\xbd\x00\x00\x08\xbe\xff\xff\xe3\xbe\xea\x91\xc9\xbd1\x8b\n\xbe\x00\x00\xe4\xbe\x00\x00\xb0\x

In [77]:
# mesh_27.show()

In [78]:
# mesh_28.show()

In [79]:
# load obj file
import numpy as np
mesh_org = trimesh.load('data/00009742_bdb26e9a610c417cbccd6c20_trimesh_019.obj')
def NDCnormalize(vertices, return_scale=False):
    """normalization in half unit ball"""
    vM = vertices.max(0)
    vm = vertices.min(0)
    scale = np.sqrt(((vM - vm) ** 2).sum(-1))
    mean = (vM + vm) / 2.0
    nverts = (vertices - mean) / scale
    if return_scale:
        return nverts, mean, scale
    return nverts
# Normalize the mesh
mesh_org.vertices[:], m, s = NDCnormalize(mesh_org.vertices, return_scale=True)
mesh_org.export('data/org_' + name.split('.')[0] + '.PLY')  # Save as PLY file

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 1816\nproperty float x\nproperty float y\nproperty float z\nelement face 3572\nproperty list uchar int vertex_indices\nend_header\n?\t\x8d<l\xaa\x9e\xbd#\x1a\xb9\xbe?\t\x8d<l\xaa\x9e\xbd#\x1a\xb9\xbeh\xdd\xfd;l\xaa\x9e\xbdz6\xfb\xbeh\xdd\xfd;l\xaa\x9e\xbdz6\xfb\xbe\x1d\xe3\x13\xb2\x95\xcc\x8e\xbdz6\xfb\xbe\x1d\xe3\x13\xb2\x95\xcc\x8e\xbdz6\xfb\xbe?\t\x8d<l\xaa\x9e\xbd\x0eW\xc6>?\t\x8d<\xbd\xec\xc1=\xf3i\xf2>?\t\x8d<\xbd\xec\xc1=\xf3i\xf2>?\t\x8d<D\t\r<\xf3i\xf2>]Km<l\xba\xb1\xbd#\x1a\xb9\xbe]Km<l\xba\xb1\xbd#\x1a\xb9\xbe\x9aZ\xea;\r\xbd\xbe\xbd#\x1a\xb9\xbe\x9aZ\xea;\r\xbd\xbe\xbd#\x1a\xb9\xbe\xb6\x92 \xbb\xdc\x90\xc1\xbd#\x1a\xb9\xbe\xb6\x92 \xbb\xdc\x90\xc1\xbd#\x1a\xb9\xbe\xe5\xb78\xbc\x10P\xb9\xbd#\x1a\xb9\xbe\xe5\xb78\xbc\x10P\xb9\xbd#\x1a\xb9\xbe\xc4R\x87\xbco\x99\xa8\xbd#\x1a\xb9\xbe\xc4R\x87\xbco\x99\xa8\xbd#\x1a\xb9\xbe\xc4R\x87\xbcj\xbb\x94\xbd#\x1a\xb9\xbe\xc4R\x87\xbcj\xbb\x94\

In [3]:
# import trimesh

# # Assume mesh_27 and mesh_28 are your trimesh.Trimesh objects

# scene = trimesh.Scene()
# scene.add_geometry(mesh_27, node_name='mesh_27')
# scene.add_geometry(mesh_28, node_name='mesh_28')
# scene.add_geometry(mesh_org, node_name='mesh_org')

# # Optionally, set different colors for each mesh
# mesh_27.visual.face_colors = [200, 0, 0, 100]  # Red, semi-transparent
# mesh_28.visual.face_colors = [0, 200, 0, 100]  # Green, semi-transparent
# mesh_org.visual.face_colors = [0, 0, 200, 10]  # Blue, semi-transparent

# scene.show()  # For inline Jupyter visualization

# Evaluation in 128 SDF

In [1]:
import gc
import os
import sys
import h5py
import fvdb
import wandb
import torch
from tqdm import tqdm
import joblib
import trimesh
import numpy as np
from skimage import measure

sys.path.append('../src/utils')
sys.path.append('../src/eval')
from ponq_eval import get_cd_f1_nc
import mesh_tools as mt
from fvdb_utils import sdfToVDB
from ssu_tools import positional_encoding


In [ ]:
import pandas as pd

class EvaluationProcessor:
    def __init__(self, 
                 abc_dir,
                 predictions_dir 
                 ):
        self.predictions_dir = predictions_dir
        self.abc_dir = abc_dir
    
    @staticmethod
    def normalize_mesh(v, target_range=1.0):
        # v: (N, 3) array of vertices
        v = np.array(v)
        centroid = v.mean(axis=0)
        v_centered = v - centroid
        max_extent = np.max(np.abs(v_centered))
        v_normalized = v_centered / max_extent * (target_range / 2)
        return v_normalized
    
    @staticmethod
    def load_ponq_mesh(ponq_data_dir, file_name, dim='64'):
        '''loads the ponq mesh from the sdf'''
        
        file = os.path.join(ponq_data_dir, f'{file_name}.hdf5')
        
        if not os.path.exists(file):
            raise FileNotFoundError(f"File {file} does not exist.")
        
        with h5py.File(file, 'r') as f:
            if f'{dim}_sdf' in f:
                sdf = f[f'{dim}_sdf'][:]
            else:
                raise ValueError(f"File {file} does not contain '{dim}_sdf' dataset.")
        
        # read the mesh
        try:
            v, f, _, _ = measure.marching_cubes(sdf, level=0)
        except Exception as e:
            print(f"Error in marching cubes for {file_name}: {e}")
            return None
        mesh = trimesh.Trimesh(vertices=v, faces=f)
        return mesh
    
    @staticmethod
    def get_prediction_mesh(pred_dir, file_name):
        # check if the file already exists
        pred_file = os.path.join(pred_dir, f'{file_name}.nvdb')
        if not os.path.exists(pred_file):
            raise FileNotFoundError(f"Prediction for {pred_file} does not exist.")
        
        # load the prediction
        # device = 'cpu'
        grid, feat, _ = fvdb.load(pred_file, device='cpu')

        # convert feature to N, F with marching cubes
        v, f, _ = grid.marching_cubes(feat)
        v = v.jdata.numpy()
        f = f.jdata.numpy()
        mesh = trimesh.Trimesh(vertices=v, faces=f)
        return mesh
    
    @staticmethod
    def mesh_from_abc(abc_dir, file_name):
        '''extracts the vertices and faces from the abc obj file'''
        
        model_dir = os.path.join(abc_dir, file_name)
        obj_files = [f for f in os.listdir(model_dir) if f.endswith('.obj')]
        file = os.path.join(model_dir, obj_files[0])
        
        if not os.path.exists(file):
            raise FileNotFoundError(f"File {file} does not exist.")
        
        # read the mesh
        mesh = trimesh.load(file)
        return mesh
    
    def get_eval(self, file_name_h5):
        '''evaluates the model on the given dataset'''
        
        print(f"Evaluating {file_name_h5}...")
        file_name = file_name_h5.split('.')[0]
        # pred_mesh = self.get_prediction_mesh(self.predictions_dir, file_name)  
        ponq_mesh = self.load_ponq_mesh(self.predictions_dir, file_name, dim='128') 
        gt_mesh = self.mesh_from_abc(self.abc_dir, file_name)

        # get matrix for evaluation using NDCnormalize
        try:
            metrics = get_cd_f1_nc((file_name, gt_mesh, ponq_mesh),
                                        scale_gt=1.0,
                                        eval_normalization=mt.NDCnormalize
                                        )
        except Exception as e:
            print(f"Error evaluating {file_name}: {e}")
            metrics = (file_name, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
        return metrics

class Evaluator:
    def __init__(self,
                 test_loader,
                 abc_dir,
                 save_predictions_dir, 
                 n_job):
        
        self.test_loader = test_loader
        self.abc_dir = abc_dir
        self.save_predictions_dir = save_predictions_dir

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # helper class for SDF to VDB conversion
        self.sdf_to_vdb = sdfToVDB()

        self.n_job = n_job

    def get_target_grid(self, input): 
        return self.sdf_to_vdb.custom_subdivide_grid(input.grid)

    def eval_sub_step(self, input):
        inputs = positional_encoding(input, self.pos_enc_dim)
        tragets_grid = self.get_target_grid(input)

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(inputs, tragets_grid)
        return outputs
    
    def save_predictions(self):
        '''saving predictions one by one'''
        file_names = []
        for batch in tqdm(self.test_loader, desc='Evaluating'):
            obj_names, _, _, _ = batch
            obj_name = obj_names[0]
            obj_name = obj_name.split('.')[0]
            
            file_names.append(obj_name)
    
        return file_names
    
    def evaluate(self):
        # print(f"Checking predictions in dir: {self.save_predictions_dir}/{self.model_name}/{self.upsampling_level}")
        test_names = self.save_predictions()

        # clean memory
        torch.cuda.empty_cache()
        gc.collect()

        # evaluate the predictions
        print(f"Evaluating predictions")
        def eval_wrapper(args):
            file_name_h5 = args
            processor = EvaluationProcessor(abc_dir=self.abc_dir, 
                                            predictions_dir=self.save_predictions_dir)
            return (file_name_h5, processor.get_eval(file_name_h5))
        def process_batch(batch):
            return [eval_wrapper(x) for x in batch]
        batch_size = 10
        batches = [test_names[i:i+batch_size] for i in range(0, len(test_names), batch_size)]
        out = joblib.Parallel(n_jobs=self.n_job)(joblib.delayed(process_batch)(batch) for batch in batches)

        out = [_item for sublist in out for _item in sublist]
        names = [_item[0] for _item in out]
        out = [_item[1] for _item in out]

        out = np.array(out)
        cd1 = out[:, 1].astype(float).mean(axis=0)
        cd2 = out[:, 2].astype(float).mean(axis=0)
        f1 = out[:, 3].astype(float).mean(axis=0)
        nc = out[:, 4].astype(float).mean(axis=0)
        ecd2 = out[:, 5].astype(float).mean(axis=0)
        ef1 = out[:, 6].astype(float).mean(axis=0)

        # table 
        df_results = pd.DataFrame({'names': names,
                                   'cd1 (x 1e-5)': out[:, 1].astype(float)* 1e5,
                                   'cd2 (x 1e-5)': out[:, 2].astype(float)* 1e5,
                                   'f1': out[:, 3].astype(float),
                                   'nc': out[:, 4].astype(float),
                                   'ecd2': out[:, 5].astype(float),
                                   'ef1': out[:, 6].astype(float)})
        # print(df_results.head())
        # save results to logger
        columns = ['CD1 (x 1e-5)', 'CD2 (x 1e-5)', 'F1', 'NC', 'ECD', 'EF1']
        data = [[cd1 * 1e5, cd2 * 1e5, f1, nc, ecd2, ef1]]
        # self.logger.log({'Evaluation': wandb.Table(data=data, columns=columns)})

        print(f"cd1: {cd1 * 1e5:.3f}, cd2: {cd2 * 1e5:.3f}, f1: {f1:.3f}, nc: {nc:.3f}, ecd2: {ecd2:.3f}, ef1: {ef1:.3f}")

        return df_results

In [42]:
config['data']['input_dir']

'/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt'

In [80]:
config['eval']['eval_dir']

'/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs'

In [43]:
evaluator = Evaluator(test_loader=test_dataloader,
                      abc_dir=config['eval']['eval_dir'],
                      save_predictions_dir=config['data']['input_dir'],
                      n_job=-1)

In [44]:
df_results = evaluator.evaluate()

Evaluating: 100%|██████████| 1071/1071 [01:31<00:00, 11.66it/s]


Evaluating predictions
Evaluating 00008322...
Evaluating 00005112...
Evaluating 00004902...
Evaluating 00003557...
Evaluating 00002580...
Evaluating 00004800...
Evaluating 00001475...
Evaluating 00000980...
Evaluating 00009354...
Evaluating 00009769...
Evaluating 00006007...
Evaluating 00008687...
Evaluating 00005754...
Evaluating 00004067...
Evaluating 00009317...
Evaluating 00007223...
Evaluating 00009041...
Evaluating 00005048...
Evaluating 00007808...
Evaluating 00005993...
Evaluating 00000106...
Evaluating 00002426...
Evaluating 00005082...
Evaluating 00003912...
Evaluating 00008204...
Evaluating 00007467...
Evaluating 00002515...
Evaluating 00008450...
Evaluating 00007826...
Evaluating 00005036...
Evaluating 00003240...
Evaluating 00005982...
Evaluating 00006132...
Evaluating 00001855...
Evaluating 00001749...
Evaluating 00008160...
Evaluating 00005570...
Evaluating 00008807...
Evaluating 00001403...
Evaluating 00001559...
Evaluating 00006316...
Evaluating 00006283...
Evaluating 

In [46]:
df_results.describe()

,cd1 (x 1e-5),cd2 (x 1e-5),f1,nc,ecd2,ef1
count,1071.000000,1071.000000,1071.000000,1071.000000,1071.000000,1071.000000
mean,479.981397,13.249502,0.857871,0.971521,0.024818,0.704086
std,1285.354717,233.206981,0.137650,0.035886,0.077254,0.418079
min,144.157364,0.123694,0.000000,0.516878,0.000000,0.000000
25%,301.452815,0.578584,0.796772,0.966979,0.000000,0.180341
50%,358.238644,0.819178,0.897347,0.980545,0.000000,1.000000
75%,434.613126,1.215879,0.955167,0.988476,0.008751,1.000000
max,38135.397784,7473.673974,0.999780,0.999600,1.006687,1.000000


In [47]:
# get row with max cd1
df_results.loc[df_results['cd1 (x 1e-5)'].idxmax()]

names               00007347
cd1 (x 1e-5)    38135.397784
cd2 (x 1e-5)     7473.673974
f1                       0.0
nc                  0.848304
ecd2                0.258271
ef1                      0.0
Name: 804, dtype: object

In [66]:
df_results.loc[df_results['cd1 (x 1e-5)'].idxmin()]


names             00009742
cd1 (x 1e-5)    144.157364
cd2 (x 1e-5)      0.123694
f1                0.998679
nc                0.980804
ecd2              0.038238
ef1               0.188981
Name: 221, dtype: object